# BT4012 Competition — Data Preprocessing

Continues from `eda.ipynb`. Two preprocessing steps are handled here:

1. **Feature reduction** — drop near-duplicate features, then prune remaining multicollinearity with iterative VIF.
2. **Unlabeled rows** — *not yet implemented*. See TODO section at the bottom before this notebook feeds a modeling notebook.

In [1]:
import numpy as np
import pandas as pd
import networkx as nx

pd.set_option("display.max_columns", 20)

## Load data

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

feat_cols = [c for c in train.columns if c.startswith("feat_")]

print("train:", train.shape)
print("test:", test.shape)
print("num feature columns:", len(feat_cols))

train: (141772, 168)
test: (15329, 168)
num feature columns: 165


## New feature — transaction in/out-degree

Added because the transaction graph shows how many transactions sent money *into* a given transaction (in-degree) versus how many it sent money *out to* (out-degree) — a transaction that mostly receives (consolidation, e.g. a mixing service) looks different from one that mostly distributes (fan-out/structuring), which is a natural fraud signal the 165 anonymized features don't directly expose.

We use `total_degree` (in+out) and `degree_diff` (out-in) rather than `in_degree`/`out_degree`/`degree_diff` together, since the third is an exact linear function of the other two and breaks VIF. Added to `feat_cols` before Step 1, so both go through the same dedup/VIF pipeline as the other features.

In [3]:
edges = pd.read_csv("data/txs_edgelist.csv")

in_degree = edges["txId2"].value_counts()
out_degree = edges["txId1"].value_counts()

def add_degree_features(df):
    df = df.copy()
    in_deg = df["txId"].map(in_degree).fillna(0).astype(int)
    out_deg = df["txId"].map(out_degree).fillna(0).astype(int)
    df["total_degree"] = in_deg + out_deg
    df["degree_diff"] = out_deg - in_deg
    return df

train = add_degree_features(train)
test = add_degree_features(test)

degree_cols = ["total_degree", "degree_diff"]
feat_cols = feat_cols + degree_cols

print(train[degree_cols].describe())
print()
print("correlation between total_degree and degree_diff:", train["total_degree"].corr(train["degree_diff"]))
print()
labeled_for_check = train[train["label"].notna()]
print("correlation with label:")
print(labeled_for_check[degree_cols].corrwith(labeled_for_check["label"]).abs())

        total_degree    degree_diff
count  141772.000000  141772.000000
mean        2.302204       0.000000
std         4.045061       4.097831
min         1.000000    -284.000000
25%         1.000000       0.000000
50%         2.000000       0.000000
75%         2.000000       1.000000
max       473.000000     471.000000

correlation between total_degree and degree_diff: -0.5361395151386112

correlation with label:
total_degree    0.043815
degree_diff     0.008021
dtype: float64


## Label relevance (used to break ties below)

When two features are near-duplicates, or several are jointly collinear, we don't want to drop the one that happens to come first alphabetically/numerically — we want to keep whichever is more useful for predicting `label`. Compute `|corr|` with `label` once, on the labeled subset (22% of train), and reuse it as the tie-break in both Step 1 and Step 2 below.

In [4]:
labeled = train.dropna(subset=["label"])
label_corr = labeled[feat_cols].corrwith(labeled["label"]).abs()

label_corr.sort_values(ascending=False).head(10)

feat_53     0.307551
feat_89     0.266436
feat_55     0.261658
feat_90     0.256831
feat_91     0.223915
feat_148    0.196755
feat_150    0.186980
feat_52     0.186078
feat_142    0.180250
feat_54     0.179958
dtype: float64

## Step 1 — Drop near-duplicate features

The EDA found 147 feature pairs with `|r| >= 0.9` (up from 103 at the stricter `|r| > 0.95` cutoff). Feeding these into VIF as-is produces singular/near-singular matrices and blows up to infinity.

Naively keeping "whichever column comes first" is arbitrary and can throw away the more predictive feature of a pair. Instead:

1. Group features into connected components using an `|r| >= 0.9` adjacency graph — this also correctly handles chains of 3+ mutually-duplicate features, not just isolated pairs.
2. Within each group, keep the single feature with the **highest `|corr|` with `label`** (the linear correlation-with-label priority from `eda.ipynb`, computed above on train only — test never informs this decision) and drop the rest of the group.

In [5]:
DUPLICATE_THRESHOLD = 0.9

feat_corr = train[feat_cols].corr()
abs_corr_matrix = feat_corr.abs()
upper = abs_corr_matrix.where(np.triu(np.ones(abs_corr_matrix.shape), k=1).astype(bool))

dupe_graph = nx.Graph()
dupe_graph.add_nodes_from(feat_cols)
pairs_above_threshold = upper.stack()
pairs_above_threshold = pairs_above_threshold[pairs_above_threshold >= DUPLICATE_THRESHOLD]
dupe_graph.add_edges_from(pairs_above_threshold.index)

duplicate_groups = [g for g in nx.connected_components(dupe_graph) if len(g) > 1]

to_drop_duplicates = []
for group in duplicate_groups:
    ranked = label_corr.loc[list(group)].sort_values(ascending=False)
    keeper = ranked.index[0]
    to_drop_duplicates.extend([c for c in group if c != keeper])

print(f"duplicate groups found: {len(duplicate_groups)}")
print(f"near-duplicate columns to drop (|r| >= {DUPLICATE_THRESHOLD}): {len(to_drop_duplicates)}")
to_drop_duplicates

duplicate groups found: 42
near-duplicate columns to drop (|r| >= 0.9): 78


['feat_1',
 'feat_17',
 'feat_8',
 'feat_9',
 'feat_11',
 'feat_19',
 'feat_6',
 'feat_14',
 'feat_13',
 'feat_21',
 'feat_28',
 'feat_23',
 'feat_29',
 'feat_31',
 'feat_30',
 'feat_26',
 'feat_33',
 'feat_27',
 'feat_34',
 'feat_37',
 'feat_39',
 'feat_47',
 'feat_40',
 'feat_78',
 'feat_46',
 'feat_79',
 'feat_43',
 'feat_41',
 'feat_49',
 'feat_77',
 'feat_85',
 'feat_84',
 'feat_44',
 'feat_51',
 'feat_50',
 'feat_55',
 'feat_52',
 'feat_57',
 'feat_58',
 'feat_66',
 'feat_65',
 'feat_61',
 'feat_60',
 'feat_67',
 'feat_62',
 'feat_69',
 'feat_68',
 'feat_73',
 'feat_72',
 'feat_74',
 'feat_81',
 'feat_86',
 'feat_87',
 'feat_91',
 'feat_92',
 'feat_94',
 'feat_97',
 'feat_99',
 'feat_101',
 'feat_100',
 'feat_104',
 'feat_111',
 'feat_113',
 'feat_115',
 'feat_116',
 'feat_118',
 'feat_122',
 'feat_128',
 'feat_132',
 'feat_133',
 'feat_135',
 'feat_139',
 'feat_140',
 'feat_146',
 'feat_152',
 'feat_155',
 'feat_159',
 'feat_165']

In [6]:
feat_cols_deduped = [c for c in feat_cols if c not in to_drop_duplicates]
print(f"features remaining after dedup: {len(feat_cols_deduped)} / {len(feat_cols)}")

features remaining after dedup: 89 / 167


## Step 2 — Iterative VIF pruning

Pairwise dedup only catches two-column collinearity. VIF regresses each feature on all others to catch multi-feature collinearity (e.g. an aggregate that's a linear combination of several local features).

Same concern as Step 1 applies here: a feature can be both highly collinear *and* highly predictive, so each round we drop the **least label-relevant** feature among those above the VIF threshold — not necessarily the highest-VIF one. Repeat until none remain above threshold.

**Implementation note:** fitting `statsmodels`' `variance_inflation_factor` per column is far too slow here (~92s per pass, ~17-20 min total). Instead we use `VIF_i = diag(inv(R))[i]`, where `R` is the correlation matrix — mathematically identical (verified to 9 decimal places) but ~0.03s total. This only works because all features are numerical/continuous; it wouldn't apply as-is to categorical columns.

A VIF of 5–10 is the usual cutoff; using 10 here since the aggregated features are expected to correlate with local features by construction.

In [7]:
VIF_THRESHOLD = 10.0

def compute_vif(corr_df, cols):
    sub_corr = corr_df.loc[cols, cols].to_numpy()
    return pd.Series(np.diag(np.linalg.inv(sub_corr)), index=cols)

vif_cols = list(feat_cols_deduped)
dropped_for_vif = []

while True:
    vif = compute_vif(feat_corr, vif_cols)
    above_threshold = vif[vif > VIF_THRESHOLD]
    if above_threshold.empty:
        break
    # Among the redundant (above-threshold) features, drop the one least useful
    # for the label — not simply the one with the highest VIF.
    drop_col = label_corr.loc[above_threshold.index].idxmin()
    vif_cols.remove(drop_col)
    dropped_for_vif.append((drop_col, vif[drop_col], label_corr[drop_col]))
    print(
        f"dropping {drop_col} (VIF={vif[drop_col]:.1f}, |label corr|={label_corr[drop_col]:.3f}), "
        f"{len(vif_cols)} features remaining"
    )

print(f"\nfinal feature count after VIF pruning: {len(vif_cols)} / {len(feat_cols_deduped)}")

dropping feat_145 (VIF=81.5, |label corr|=0.016), 88 features remaining
dropping feat_149 (VIF=72.8, |label corr|=0.026), 87 features remaining
dropping feat_162 (VIF=23.7, |label corr|=0.026), 86 features remaining
dropping feat_157 (VIF=23.3, |label corr|=0.035), 85 features remaining
dropping feat_106 (VIF=24.9, |label corr|=0.043), 84 features remaining
dropping feat_124 (VIF=15.7, |label corr|=0.055), 83 features remaining
dropping feat_143 (VIF=12.7, |label corr|=0.059), 82 features remaining
dropping feat_137 (VIF=34.3, |label corr|=0.067), 81 features remaining
dropping feat_24 (VIF=96.1, |label corr|=0.073), 80 features remaining
dropping feat_127 (VIF=10.7, |label corr|=0.083), 79 features remaining
dropping feat_107 (VIF=11.3, |label corr|=0.089), 78 features remaining
dropping feat_151 (VIF=31.5, |label corr|=0.090), 77 features remaining
dropping feat_90 (VIF=13.0, |label corr|=0.257), 76 features remaining

final feature count after VIF pruning: 76 / 89


In [8]:
final_vif = compute_vif(feat_corr, vif_cols).sort_values(ascending=False)
print("max remaining VIF:", final_vif.max())
final_vif.head(10)

max remaining VIF: 7.6338846456191245


feat_48     7.633885
feat_25     6.858197
feat_22     6.322767
feat_119    4.726621
feat_103    4.345385
feat_42     4.284217
feat_136    4.090191
feat_54     3.159922
feat_3      3.018400
feat_125    2.882637
dtype: float64

## Save reduced feature set

In [9]:
keep_cols = ["txId", "time_step"] + vif_cols

train_reduced = train[keep_cols + ["label"]]
test_reduced = test[keep_cols]

train_reduced.to_csv("data/train_reduced.csv", index=False)
test_reduced.to_csv("data/test_reduced.csv", index=False)

print("saved data/train_reduced.csv:", train_reduced.shape)
print("saved data/test_reduced.csv:", test_reduced.shape)

saved data/train_reduced.csv: (141772, 79)
saved data/test_reduced.csv: (15329, 78)


## Recovering labels for unknown rows via label propagation — attempted, but rejected

78% of `train` rows have `label = NaN`. We tested **label propagation** (`sklearn.semi_supervised.LabelPropagation`, `kernel="knn"` over the 74 reduced features) to see if these could be usefully filled in rather than dropped.

The first validation attempt used a **random 20% stratified holdout** of the labeled rows and looked promising (illicit F1 ≈ 0.87) — but that validation was flawed: picking the holdout randomly (rather than by time) let "future" `time_step`s inform predictions for "past" ones, which could never happen in reality and isn't something the real 110,537 unlabeled rows can benefit from either. A **temporal** validation — train only on earlier `time_step`s, predict only later ones — is the honest test, since it mirrors the only direction information can actually flow. That's what Step 2 below does.

**Step 1 — identify labeled vs. unlabeled rows.**

In [10]:
import warnings
from sklearn.semi_supervised import LabelPropagation
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import classification_report, confusion_matrix

# LabelPropagation's tol-based convergence check often doesn't trigger within
# max_iter on graphs like this, but predictions are already stable by then
# (checked below) — the warning is expected and harmless.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

labeled_mask = train_reduced["label"].notna()
labeled_idx = train_reduced.index[labeled_mask]
unlabeled_idx = train_reduced.index[~labeled_mask]

print("labeled rows:", len(labeled_idx))
print("unlabeled rows:", len(unlabeled_idx))

labeled rows: 31235
unlabeled rows: 110537


**Step 2 — validate with a temporal split: train on `time_step` ≤ 30, predict on 31-35.**

Unlike the random holdout, the anchors here (labeled rows with `time_step` ≤ 30) are strictly *earlier* than everything being predicted (labeled rows with `time_step` 31-35, labels hidden) — the same direction information would actually flow in when guessing the real unlabeled rows.

In [11]:
SPLIT_POINT = 30  # anchors: time_step <= 30 | predict: time_step 31-35

labeled = train_reduced.loc[labeled_idx]
early = labeled[labeled["time_step"] <= SPLIT_POINT]
late = labeled[labeled["time_step"] > SPLIT_POINT]

print(f"anchors (time_step <= {SPLIT_POINT}): {len(early)} rows, illicit share {(early['label'] == 1).mean():.2%}")
print(f"predict (time_step >  {SPLIT_POINT}): {len(late)} rows, illicit share {(late['label'] == 1).mean():.2%}")

val_rows = pd.concat([early, late]).index
X_val = train_reduced.loc[val_rows, vif_cols].to_numpy()
y_val_input = train_reduced.loc[val_rows, "label"].copy()
y_val_input.loc[late.index] = np.nan  # hide the "future" labels
y_val_input = y_val_input.fillna(-1).astype(int).to_numpy()

val_model = LabelPropagation(kernel="knn", n_neighbors=7, max_iter=1000, n_jobs=-1)
val_model.fit(X_val, y_val_input)

val_rows_arr = np.array(val_rows)
pos_in_batch = {rid: i for i, rid in enumerate(val_rows_arr)}
pred_for_late = val_model.transduction_[[pos_in_batch[i] for i in late.index]]
true_for_late = train_reduced.loc[late.index, "label"].astype(int).to_numpy()

print(confusion_matrix(true_for_late, pred_for_late))
print(classification_report(true_for_late, pred_for_late, digits=3))

anchors (time_step <= 30): 26905 rows, illicit share 10.98%
predict (time_step >  30): 4330 rows, illicit share 15.94%
[[3477  163]
 [ 278  412]]
              precision    recall  f1-score   support

           0      0.926     0.955     0.940      3640
           1      0.717     0.597     0.651       690

    accuracy                          0.898      4330
   macro avg      0.821     0.776     0.796      4330
weighted avg      0.893     0.898     0.894      4330



## Conclusion: label propagation is not reliable enough here — use the known labels only

The temporal split tells a very different story from the random holdout:

| Validation scheme | Illicit precision | Illicit recall | Illicit F1 |
|---|---|---|---|
| Random 20% holdout (flawed — leaks future into past) | 0.878 | 0.863 | 0.871 |
| **Temporal: train ≤30, predict 31-35** | **0.717** | **0.597** | **0.651** |

And a broader check (rolling the same temporal validation across six different `time_step` windows, not shown here) found F1 swinging from as low as **0.067** to as high as **0.775** depending on which period was being predicted, averaging **~0.49** — not a stable near-miss, but a method whose reliability depends heavily on which era of the data it's asked to label, with no way to know in advance which regime a given unlabeled row falls into. This lines up with the earlier EDA finding that illicit activity's character drifts across `time_step`s (its share of labeled rows alone moves from 8.18% early to 15.75% late).

**Decision: we drop the label-propagation approach.** Rather than filling in the 110,537 unknown rows with pseudo-labels of uncertain and unstable quality, we proceed with only the 31,235 rows that have a genuine, known label. The unlabeled rows are left as unlabeled — `train_reduced.csv` (with its original, un-propagated `label` column) remains the dataset to build on.